# How the Image Matching Challenge 2025 Score Is Computed

**Clustering-aware camera-center mean Average Accuracy (mAA), with a fully worked similarity-transform alignment.**

This notebook explains, end to end, how a submission to the *Image Matching Challenge 2025* is scored, and works through concrete numerical examples with LaTeX equations, tables, and runnable code that reproduces every number.

Sources:
- [Image Matching Challenge 2025 (Ongoing) — Kaggle](https://www.kaggle.com/competitions/image-matching-challenge-2025-ongoing/overview)
- [A recap of the 2025 Image Matching Challenge — Kaggle blog](https://www.kaggle.com/blog/a-recap-of-the-2025-image-matching-challenge)
- [IMC 2024 Methods & Solutions Review (arXiv:2407.03172) — camera-center mAA](https://arxiv.org/html/2407.03172)
- [Mismatched: Evaluating the Limits of Image Matching (arXiv:2408.16445)](https://arxiv.org/html/2408.16445v1)


## 1. Two different IMC metric families (do not confuse them)

IMC has used two *different* scoring families over the years:

- **Relative-pose metric (IMC ~2019–2023).** Works on image **pairs**. For each pair it computes a rotation error and a translation(-angle) error, thresholds them (the familiar
$\max(\text{err}_{\text{angle}}, \text{err}_{\text{translation}}) < \tau$ idea), and averages. There is no clustering and no similarity transform.

- **Camera-center metric (IMC 2024 and 2025).** Works on **individual cameras**. It aligns your camera *centers* to the ground-truth centers with one global similarity transform, then a camera is *registered* if its center lands within a **distance** threshold. This is where "fraction of registered cameras" comes from.

> **IMC 2025 uses the camera-center metric**, plus **clustering** on top. There is *no* $\max(\text{rotation}, \text{translation})$ step in 2025 — the threshold is a **Euclidean distance between 3D camera centers** after alignment.


## 2. What you submit

A **dataset** is a bag of images from *one or more* scenes, plus some **outlier** images that belong to no scene. Your job:

1. **Cluster** the images into scenes.
2. **Register** each image — estimate its pose (rotation $R$, translation $t$; equivalently the camera center $C = -R^\top t$). Images you cannot place are left **unregistered**.

Your submission is a table:

| dataset | scene (= your cluster label) | image | rotation_matrix | translation_vector |
|---------|------------------------------|-------|-----------------|--------------------|

The **scene column is your clustering**; the **rotation/translation columns are your registration**. Both come from you (typically from a Structure-from-Motion pipeline: feature matching → COLMAP / GLOMAP).


## 3. The camera-center mAA (the pose part)

Your reconstruction lives in an **arbitrary coordinate frame** — arbitrary origin, orientation, and scale — because SfM is only defined up to a similarity. So the scorer first finds the best **similarity transform** $T$ mapping your centers $C$ onto the hidden ground-truth centers $C_g$:

$$
T(C) = s\,R\,C + \mathbf{t}, \qquad s \in \mathbb{R}_{>0},\; R \in SO(3),\; \mathbf{t} \in \mathbb{R}^3 .
$$

$T$ is found by minimizing the alignment error over the corresponding centers (Umeyama / Horn closed form), robustly via a RANSAC-like search over triplets (keep the model with the most inliers):

$$
T^\star = \arg\min_{s,R,\mathbf{t}} \sum_{i} \big\lVert\, C_{g,i} - (s\,R\,C_i + \mathbf{t}) \,\big\rVert^2 .
$$

A camera is **registered / accurate at threshold** $\tau$ if

$$
\big\lVert\, C_{g,i} - T^\star(C_i) \,\big\rVert < \tau .
$$

For a scene with $N$ ground-truth cameras, the **accuracy at threshold** $\tau$ is a *fraction*:

$$
A(\tau) = \frac{\#\{\, i : \lVert C_{g,i} - T^\star(C_i)\rVert < \tau \,\}}{N},
$$

the **Average Accuracy** over a set of thresholds $\mathcal{T}$ is

$$
\mathrm{AA} = \frac{1}{|\mathcal{T}|}\sum_{\tau \in \mathcal{T}} A(\tau),
$$

and the **dataset / final score** averages AA over scenes and then over datasets:

$$
\text{score}_{\text{dataset}} = \frac{1}{|\mathcal{S}|}\sum_{s \in \mathcal{S}} \mathrm{AA}_s,
\qquad
\text{score}_{\text{final}} = \frac{1}{|\mathcal{D}|}\sum_{d \in \mathcal{D}} \text{score}_d .
$$

Unregistered or wrongly-clustered cameras stay in the denominator $N$ but never enter the numerator — they can only lower the score.


## 4. Folding in clustering (the 2025 extension)

Predicted clusters are matched to ground-truth scenes with a **best assignment** (one predicted cluster per scene) that maximizes total score. For a scene $s$ mapped to cluster $P$:

- The transform $T^\star$ is fit using only images in $P \cap s$ (in both your cluster **and** the true scene).
- Cameras of $s$ that you placed in a *different* cluster are **never eligible** — automatic failures.
- Cameras in $P$ that belong to a *different* scene (impurities) or are outliers are **not** in $s$'s ground truth — they earn nothing and can perturb the fit.

So clustering **gates eligibility**, registration **gates accuracy**. One misclustered camera can hurt two scenes (missing from its own, dead weight in another).


## 5. Worked example A — clustering effects

Ground truth: **Scene A** $=\{a_1,\dots,a_6\}$ (6 cams), **Scene B** $=\{b_1,\dots,b_4\}$ (4 cams), **$o_1$** = outlier.

Your (deliberately messy) clustering:

| cluster | members | note |
|---------|---------|------|
| $C_1$ | $\{a_1,a_2,a_3,a_4,\,b_1\}$ | 4 from A + impurity $b_1$ |
| $C_2$ | $\{a_5,a_6\}$ | rest of Scene A, split off |
| $C_3$ | $\{b_2,b_3,b_4,\,o_1\}$ | 3 from B + outlier $o_1$ |

**Best mapping:** Scene A $\to C_1$ (4 A-cams > $C_2$'s 2); Scene B $\to C_3$ (3 B-cams > the single $b_1$ in $C_1$). $C_2$ is left unmatched.

Distance thresholds $\mathcal{T} = \{0.1, 0.25, 0.5, 1.0\}$ (normalized units; the real metric uses more).


**Scene A** — 6 GT cameras, mapped to $C_1$. Transform fit from $C_1 \cap A = \{a_1,a_2,a_3,a_4\}$. $a_5,a_6$ (in $C_2$) are never eligible; $b_1$ is an impurity. Aligned center errors: $a_1=0.05,\ a_2=0.2,\ a_3=0.6,\ a_4=2.5$. Fraction is out of **all 6**:

| $\tau$ | centers within $\tau$ | fraction |
|--------|-----------------------|----------|
| 0.10 | $\{a_1\}$ | $1/6 = 0.167$ |
| 0.25 | $\{a_1,a_2\}$ | $2/6 = 0.333$ |
| 0.50 | $\{a_1,a_2\}$ | $2/6 = 0.333$ |
| 1.00 | $\{a_1,a_2,a_3\}$ | $3/6 = 0.500$ |

$$\mathrm{AA}(A) = \tfrac{1}{4}(0.167+0.333+0.333+0.500) = 0.333$$

**Scene B** — 4 GT cameras, mapped to $C_3$. Transform fit from $C_3 \cap B = \{b_2,b_3,b_4\}$; $b_1$ is lost (wrong cluster), $o_1$ is an outlier. Errors: $b_2=0.08,\ b_3=0.15,\ b_4=0.9$. Fraction out of **4**:

| $\tau$ | centers within $\tau$ | fraction |
|--------|-----------------------|----------|
| 0.10 | $\{b_2\}$ | $1/4 = 0.25$ |
| 0.25 | $\{b_2,b_3\}$ | $2/4 = 0.50$ |
| 0.50 | $\{b_2,b_3\}$ | $2/4 = 0.50$ |
| 1.00 | $\{b_2,b_3,b_4\}$ | $3/4 = 0.75$ |

$$\mathrm{AA}(B) = \tfrac{1}{4}(0.25+0.50+0.50+0.75) = 0.50$$

$$\text{score}_{\text{dataset}} = \tfrac{1}{2}(0.333 + 0.50) = 0.417$$


In [1]:
import numpy as np

def average_accuracy(errors, N, thresholds):
    "AA = mean over thresholds of (#cameras with error < tau) / N."
    errors = np.asarray(errors, dtype=float)
    per_tau = [np.sum(errors < t) / N for t in thresholds]
    return per_tau, float(np.mean(per_tau))

thresholds = [0.1, 0.25, 0.5, 1.0]

# Scene A: 6 GT cams; only a1..a4 eligible (a5,a6 in wrong cluster -> treat as inf error)
errA = [0.05, 0.2, 0.6, 2.5, np.inf, np.inf]
perA, aaA = average_accuracy(errA, N=6, thresholds=thresholds)

# Scene B: 4 GT cams; b1 in wrong cluster -> inf; b2,b3,b4 eligible
errB = [np.inf, 0.08, 0.15, 0.9]
perB, aaB = average_accuracy(errB, N=4, thresholds=thresholds)

print("Scene A per-threshold fractions:", [round(x,3) for x in perA], " AA(A) =", round(aaA,3))
print("Scene B per-threshold fractions:", [round(x,3) for x in perB], " AA(B) =", round(aaB,3))
print("dataset score =", round((aaA + aaB)/2, 3))


Scene A per-threshold fractions: [np.float64(0.167), np.float64(0.333), np.float64(0.333), np.float64(0.5)]  AA(A) = 0.333
Scene B per-threshold fractions: [np.float64(0.25), np.float64(0.5), np.float64(0.5), np.float64(0.75)]  AA(B) = 0.5
dataset score = 0.417


## 6. Worked example B — the similarity-transform alignment (Umeyama / Horn)

Real IMC is 3D; here we use 2D camera centers so every number is checkable by hand. The 3D case is identical with a $3\times 3$ rotation.

**Ground-truth centers** (hidden, meters), 5 cameras:

$$C_{g,1}=(10,5),\ C_{g,2}=(10,7),\ C_{g,3}=(8,7),\ C_{g,4}=(8,5),\ C_{g,5}=(9,6)$$

**Your SfM estimates** (arbitrary frame). Cameras 1–4 have perfect *relative* geometry (a unit square); camera 5 you misplaced:

$$C_1=(0,0),\ C_2=(1,0),\ C_3=(1,1),\ C_4=(0,1),\ C_5=(0.5,\,2.0)$$

RANSAC selects cameras 1–4 as the consensus set (they agree perfectly); the transform is fit from them and camera 5 is only *measured*.


### Step 1 — Centroids (of the 4 inliers)
$$\mu_C = \tfrac14\!\sum_{i=1}^4 C_i = (0.5,\,0.5), \qquad \mu_G = \tfrac14\!\sum_{i=1}^4 C_{g,i} = (9,\,6)$$

### Step 2 — Center both point sets
$$C_1'=(-0.5,-0.5),\ C_2'=(0.5,-0.5),\ C_3'=(0.5,0.5),\ C_4'=(-0.5,0.5)$$
$$C_{g,1}'=(1,-1),\ C_{g,2}'=(1,1),\ C_{g,3}'=(-1,1),\ C_{g,4}'=(-1,-1)$$

### Step 3 — Cross-covariance $M=\sum_i C_{g,i}'\,(C_i')^\top$
$$M = \begin{bmatrix} 0 & -2 \\ 2 & 0 \end{bmatrix}$$

### Step 4 — Rotation from the SVD $M=U\Sigma V^\top$
$$M = 2\begin{bmatrix}0&-1\\1&0\end{bmatrix} \Rightarrow U=\begin{bmatrix}0&-1\\1&0\end{bmatrix},\ \Sigma=\mathrm{diag}(2,2),\ V^\top=I$$
$$R = U V^\top = \begin{bmatrix}0&-1\\1&0\end{bmatrix} \quad (\text{a } 90^\circ \text{ CCW rotation})$$
In 3D, $R = U\,\mathrm{diag}(1,\dots,1,\det(UV^\top))\,V^\top$ — the last term guards against a reflection.

### Step 5 — Scale
$$s = \frac{\operatorname{tr}(\Sigma)}{\sum_i \lVert C_i' \rVert^2} = \frac{2+2}{0.5+0.5+0.5+0.5} = \frac{4}{2} = 2$$

### Step 6 — Translation
$$\mathbf{t} = \mu_G - s\,R\,\mu_C = (9,6) - 2\,R\,(0.5,0.5) = (9,6) - (-1,1) = (10,5)$$

**Recovered transform:** $s=2,\ R=90^\circ\text{ CCW},\ \mathbf{t}=(10,5)$.


### Step 7 — Apply $T(C)=sRC+\mathbf{t}$ and measure residuals

| cam | your $C$ | $T(C)$ | true $C_g$ | residual $\lVert C_g - T(C)\rVert$ |
|-----|----------|--------|-----------|-----------------------------------|
| 1 | $(0,0)$ | $(10,5)$ | $(10,5)$ | **0** |
| 2 | $(1,0)$ | $(10,7)$ | $(10,7)$ | **0** |
| 3 | $(1,1)$ | $(8,7)$ | $(8,7)$ | **0** |
| 4 | $(0,1)$ | $(8,5)$ | $(8,5)$ | **0** |
| 5 | $(0.5,2.0)$ | $(6,6)$ | $(9,6)$ | **3.0** |

Camera 5: $R(0.5,2.0)=(-2.0,0.5)\to \times 2=(-4,1)\to +(10,5)=(6,6)$; true is $(9,6)$, off by $3.0$. Your *relative* placement of camera 5 was wrong, so **no** global similarity can rescue it.

### Step 8 — Threshold $\to$ fraction, over 5 cameras, $\mathcal{T}=\{0.5,1,2,4\}$

Registration uses the strict inequality $\lVert C_g - T(C)\rVert < \tau$. Camera 5's residual is exactly $3.0$, so it only registers once $\tau > 3.0$:

| $\tau$ | registered | fraction |
|--------|------------|----------|
| 0.5 | $\{1,2,3,4\}$ | $4/5=0.8$ |
| 1.0 | $\{1,2,3,4\}$ | $4/5=0.8$ |
| 2.0 | $\{1,2,3,4\}$ | $4/5=0.8$ |
| 4.0 | $\{1,2,3,4,5\}$ | $5/5=1.0$ |

$$\mathrm{AA} = \tfrac14(0.8+0.8+0.8+1.0) = 0.85$$


In [2]:
import numpy as np

def umeyama(src, dst, with_scale=True):
    "Least-squares similarity T(x)=s R x + t mapping src->dst (Umeyama 1991)."
    src = np.asarray(src, float); dst = np.asarray(dst, float)
    n, d = src.shape
    mu_s, mu_d = src.mean(0), dst.mean(0)
    Sc, Dc = src - mu_s, dst - mu_d
    M = (Dc.T @ Sc) / n                    # cross-covariance
    U, D, Vt = np.linalg.svd(M)
    S = np.eye(d)
    if np.linalg.det(U @ Vt) < 0:          # reflection guard
        S[-1, -1] = -1
    R = U @ S @ Vt
    var_src = (Sc ** 2).sum() / n
    s = (D @ np.diag(S)).sum() / var_src if with_scale else 1.0
    t = mu_d - s * R @ mu_s
    return s, R, t

# Consensus set (RANSAC inliers): cameras 1-4
C  = np.array([[0,0],[1,0],[1,1],[0,1]], float)
Cg = np.array([[10,5],[10,7],[8,7],[8,5]], float)
s, R, t = umeyama(C, Cg)
print("s =", round(s,3))
print("R =\n", np.round(R,3))
print("t =", np.round(t,3))

# Apply to all 5 cameras (incl. the misplaced camera 5) and get residuals
C_all  = np.array([[0,0],[1,0],[1,1],[0,1],[0.5,2.0]], float)
Cg_all = np.array([[10,5],[10,7],[8,7],[8,5],[9,6]],    float)
T = (s * (R @ C_all.T)).T + t
res = np.linalg.norm(Cg_all - T, axis=1)
for i,(p,r) in enumerate(zip(T, res), 1):
    print(f"cam {i}: T(C)={np.round(p,2)}  residual={round(r,3)}")

thr = [0.5, 1.0, 2.0, 4.0]   # strict < ; camera 5 residual is exactly 3.0
per = [np.mean(res < tt) for tt in thr]
print("per-threshold fractions:", [round(x,2) for x in per], " AA =", round(float(np.mean(per)),3))


s = 2.0
R =
 [[ 0. -1.]
 [ 1.  0.]]
t = [10.  5.]
cam 1: T(C)=[10.  5.]  residual=0.0
cam 2: T(C)=[10.  7.]  residual=0.0
cam 3: T(C)=[8. 7.]  residual=0.0
cam 4: T(C)=[8. 5.]  residual=0.0
cam 5: T(C)=[6. 6.]  residual=3.0
per-threshold fractions: [np.float64(0.8), np.float64(0.8), np.float64(0.8), np.float64(1.0)]  AA = 0.85


## 7. Takeaways

- The alignment **removes the free gauge** (origin / orientation / scale) that every SfM reconstruction has, so only your *relative* geometry is judged — you are never penalized for reconstructing in a rotated or rescaled frame.
- **RANSAC matters:** the consensus set defines $T$; a gross outlier is excluded from the fit (otherwise least squares would smear its error across the good cameras and unfairly wreck their residuals).
- A camera's **residual** is exactly what is compared to the distance thresholds — that is where numbers like $a_1=0.05,\ a_4=2.5$ come from.
- The denominator $N$ is the scene's **true** camera count, so one bad or misclustered camera in a small scene hurts proportionally more.
- **2025 mental model:** submit clusters + poses $\Rightarrow$ scorer aligns your poses to hidden ground truth per scene $\Rightarrow$ counts the fraction of each true scene's cameras whose centers land within a distance threshold. Clustering gates eligibility; registration gates accuracy. No $\max(\text{rotation},\text{translation})$ anywhere — that was the older pairwise metric.
